In [0]:
# create text widget to get raw_path from job parameter, or default to second value
dbutils.widgets.text("raw_path", "/Volumes/workspace/tfl/raw/")
raw_path = dbutils.widgets.get("raw_path")  # get parameter from widget

# list all files at in path
files = dbutils.fs.ls(raw_path)
print(f"Found {len(files)} files in {raw_path}")

if len(files) == 0:
    raise ValueError(f"No files found in {raw_path}")


In [0]:
# find which csv files are not already in bronze table
bronze_files = {
    row.source_file.split("/")[-1]
    for row in spark.table("workspace.tfl.bronze_journeys").select("source_file").distinct().collect()
    if row.source_file is not None
}

files = [f for f in files if f.name.endswith(".csv") and f.name not in bronze_files]
print(f"{len(files)} csv files are not yet in the bronze table")

In [0]:
# define number of files as a task value which can be passed between tasks in a job
dbutils.jobs.taskValues.set(key="file_count", value=len(files))